In [3]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

## I will first load the dataset to find the sd & mean parameters to normalize with

In [6]:
temp_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224), #final input dimension
    transforms.ToTensor()
])

temp_dataset = datasets.ImageFolder('./data', transform=temp_transform)
temp_loader = DataLoader(temp_dataset, batch_size=64, shuffle=False, num_workers=4)


In [ ]:
def calculate_mean(loader):
    channel_sum = torch.zeros(3)
    channel_count = 0

    for images, _ in loader:
        channel_sum += torch.mean(images, dim=[0, 2, 3]) * images.shape[0] #means of RGB channels per batch
        channel_count += images.shape[0] # of imgs in batch
        
    # Final Mean (sum of means * number of samples / total number of samples)
    mean = channel_sum / channel_count
    return mean.tolist()

def calculate_std(mean, loader):
    mean_tensor = torch.tensor(mean).view(1, 3, 1, 1)
    
    # Initialize running sum of squared differences
    channel_std_sum = torch.zeros(3)
    channel_count = 0
    
    for images, _ in loader:
        # Calculate the difference squared: (Image - Mean)^2
        # This uses broadcasting: (B, 3, H, W) - (1, 3, 1, 1)
        diff = (images - mean_tensor).pow(2)
        
        # Sum the squared differences across H, W, and B (Batch)
        channel_std_sum += torch.sum(diff, dim=[0, 2, 3])
        channel_count += diff.shape[0] * diff.shape[2] * diff.shape[3]
        
    # The final Standard Deviation is the square root of the average squared difference (Variance)
    # We use (N - 1) for the denominator for sample standard deviation, but N (total pixels) 
    # is often used for large datasets to simplify.
    std = torch.sqrt(channel_std_sum / channel_count) 
    return std.tolist()